## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [3]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

/Users/rapido/analytics/customer/Appography/Appography_v2


### Customer installed apps

In [13]:
## Parameter 
start_date = '20240318'
end_date = '20240331'
city = 'Hyderabad'

In [14]:
##customer_installed_apps

start_time = time.time()

customer_installed_apps_query = f"""

WITH active_customers AS (

    SELECT 
        customer_id
    FROM 
        orders.order_logs_snapshot
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND channel_host IN ('android')
        AND city_name = '{city}'
    ),     
    
    customer_app AS (
    
    SELECT
        id user, 
        yyyymmdd,
        updated_epoch,
        json_extract(data, '$.appographyData') as app_list
    FROM 
        raw_internal.kafka_info_customer_installed_apps_v1
    WHERE 
        yyyymmdd >= DATE_FORMAT((DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '2' MONTH), '%Y%m%d')
        AND yyyymmdd <= DATE_FORMAT((CURRENT_DATE - INTERVAL '1' DAY), '%Y%m%d')
    )
    
    SELECT
        user,
        app_list
    FROM 
        (
        SELECT
            customer_app.*,
            row_number() over(partition by user order by updated_epoch DESC) updated_seq
        FROM 
            active_customers
        JOIN 
            customer_app
            ON customer_id = user
        )
    WHERE 
        updated_seq = 1
"""

df_customer_installed_apps = pd.read_sql(customer_installed_apps_query, connection)

df_customer_installed_apps_v1 = df_customer_installed_apps[['user', 'app_list']].copy()

##Define a function to extract values between double quotations and trim extra spaces
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r'"(.*?)"', text)]

df_customer_installed_apps_v1['app_list'] = df_customer_installed_apps_v1['app_list'].apply(lambda x: extract_and_trim(x))

end_time = time.time()
execution_time = ((end_time - start_time)/60.00)
print("Execution time:", execution_time, "mins")

Execution time: 2.0462626020113626 mins


In [15]:
df_customer_installed_apps_v1

,user,app_list
0,656d6c78be45f1519468f640,"[Aha, Amazon Prime Video, Amazon Music, Bookin..."
1,658f99ad993d04939af5627b,"[Clock, Facebook, Messenger, Flipkart, Google ..."
2,64f8adff3917fcc97b65a82c,"[Zomato, CRED, Flipkart, Google Sheets, Google..."
3,6348d12fb38d5d31230a9588,"[Wynk Music, Facebook, Messenger, Flipkart, Go..."
4,62d79c22b0c23282dd04ba4b,"[Aha, Amazon Prime Video, Zomato, BigBasket, B..."
...,...,...
1616309,5e1077c9715b3f22b965bf37,"[Adobe Lightroom, Amazon Shopping, Axis Mobile..."
1616310,62f66354c30fe22fbb8ea6a0,"[Amazon Prime Video, BookMyShow, Facebook, Fli..."
1616311,633eaea9417898d9e79927a8,"[Aha, Amazon Prime Video, Zomato, Wynk Music, ..."
1616312,5dab2970a52eca27857c0040,"[Facebook, Messenger, Google Maps, Google Phot..."


### Customer base

In [18]:
##Please replace the different customer cohort

start_time = time.time()


##active_customer_query
active_customer_demographic_query = f"""

    WITH active_customers AS (

    SELECT 
        channel_host,
        customer_id,
        COALESCE(city_name, 'NA') city_name
    FROM 
        (
        SELECT 
            channel_host,
            customer_id,
            city_name,
            row_number() over(partition by customer_id order by updated_epoch DESC) updated_seq
        FROM 
            orders.order_logs_snapshot
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND channel_host IN ('android', 'ios')
            AND city_name = '{city}'
        ) 
    WHERE
        updated_seq = 1
    ), 

    customer_single_view AS (

    SELECT
        customerId,
        gender,
        lifetimeage rapido_age,
        CASE 
        WHEN ((POSITION('/' IN dateofbirth) > 0 AND dateofbirth <> 'NaN/NaN/NaN') or POSITION('-' IN dateofbirth) > 0) 
        THEN 
            CASE 
            WHEN TRY(date_diff('year', CAST(dateofbirth AS DATE), current_date)) IS NOT NULL 
            THEN date_diff('year', CAST(dateofbirth AS DATE), current_date)
            ELSE NULL
            END
        ELSE NULL
        END AS customer_age,
        lifetimerides
    FROM 
        datasets.customer_single_view
    WHERE 
        customerId IN (SELECT customer_id FROM active_customers)
    ),

    iallocator_customer_segments AS (

    SELECT
        customer_id user_id,
        taxi_ltr_segment ltr_segment,
        taxi_lifetime_rides life_time_rides,
        taxi_lifetime_stage life_time_stage,
        taxi_income_segment income_segment,
        taxi_service_affinity service_affinity,
        taxi_distance_preference distance_preference,
        CASE 
        WHEN ps_tag_auto = ps_tag_link THEN ps_tag_auto
        WHEN ps_tag_auto = 'PS' OR ps_tag_link = 'PS' THEN 'PS'
        WHEN ps_tag_auto = 'NPS' OR ps_tag_link = 'NPS' THEN 'NPS'
        ELSE 'UNKNOWN'
        END ps_tag_taxi

    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date = DATE_FORMAT(DATE_PARSE('{end_date}', '%Y%m%d'), '%Y-%m-%d')
        AND customer_id IN (SELECT customer_id FROM active_customers)
    )

    SELECT
        a.*,
        c.gender,
        c.rapido_age,
        COALESCE(CAST(c.customer_age AS VARCHAR), 'NA') customer_age,
        CASE 
        WHEN life_time_rides = 0 OR c.lifetimerides = 0 THEN 'LTR 0'
        WHEN life_time_rides > 0 THEN ltr_segment
        ELSE 'UNKNOWN'
        END AS ltr_segment,
        COALESCE(life_time_rides, c.lifetimerides) life_time_rides,
        COALESCE(life_time_stage, 'UNKNOWN') AS life_time_stage,
        COALESCE(income_segment, 'UNKNOWN') AS income_segment,
        COALESCE(service_affinity, 'NO_AFFINITY') AS service_affinity,
        COALESCE(distance_preference, 'UNKNOWN') AS distance_preference,
        COALESCE(ps_tag_taxi, 'UNKNOWN') AS price_sensitivity 
    FROM 
        active_customers a
    LEFT JOIN 
        customer_single_view c
        ON a.customer_id = c.customerId
    LEFT JOIN 
        iallocator_customer_segments b
        ON a.customer_id = b.user_id
"""

df_active_customer_demographic = pd.read_sql(active_customer_demographic_query, connection)

active_customer_ao_net_query = f"""

    WITH fe_tbl AS (
        SELECT
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            city,
            service_name,
            user_id as user_id,
            fare_estimate_id
        FROM
            hive.pricing.fare_estimates_enriched
        WHERE 
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city = '{city}'
    ),

    rr_net_tbl as (
        SELECT   
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            city_name AS city,
            service_obj_service_name AS service_name,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag
        FROM orders.order_logs_snapshot ols
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND channel_host IN ('android', 'ios')
            AND city_name = '{city}'
    ),

    fe_merged AS (
        SELECT
            fe_tbl.*,
            rr_net_tbl.customer_id,
            rr_net_tbl.order_id,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.order_date = rr_net_tbl.order_date
            AND fe_tbl.user_id = rr_net_tbl.customer_id
    )

    SELECT
        customer_id,
        COUNT(distinct fare_estimate_id) AS fe_count,
        COUNT(distinct order_id) AS rr_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count

    FROM 
        fe_merged
    GROUP BY 1

"""

df_active_customer_ao_net_query = pd.read_sql(active_customer_ao_net_query, connection)

df_active_customer = pd.merge(df_active_customer_demographic,
                              df_active_customer_ao_net_query,
                              how='left',
                              on = ['customer_id']
                             )

end_time = time.time()
execution_time = ((end_time - start_time)/60.00)
print("Execution time:", execution_time, "mins")

Execution time: 4.197930967807769 mins


In [19]:
df_active_customer.head(2)

,channel_host,customer_id,city_name,gender,rapido_age,customer_age,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,fe_count,rr_count,net_count
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0
1,android,5d4ed35b55fbf50d45edac48,Hyderabad,MALE,1700.0,24,PHH,80,SUSTENANCE,MEDIUM_INCOME,NO_AFFINITY,UNKNOWN,NPS,5.0,3.0,3.0


### category list

Google Sheet - https://docs.google.com/spreadsheets/d/1O6IsJ2dzzhXKuoHFoOA0AD4s4SOerMC1U-KrpeFnbwQ/edit#gid=1742529009

In [ ]:
##Read the shared mapping csv file

"""
cwd = os.getcwd()
file_path = os.path.join(cwd, 'category_list_v1.csv')
category_list = pd.read_csv(file_path)

"""

##category_list = pd.read_clipboard()

In [21]:
category_list['app_name'] = category_list['app_name'].str.lower()
category_list = category_list[['app_name', 'categories_l2', 'categories_l1']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(399, 3)
['Educational' 'OTT' 'Delivery_Grocery' 'Tools' 'Office' 'Dating' 'News'
 'Devotional' 'Travel_Metro' 'Gaming' 'Travel_Bookings'
 'Finance_Transactions' 'Ecommerce' 'Streaming_Music' 'Finance_Investment'
 'Health' 'Fantasy_Sports' 'Delivery_Food' 'Home_Security' 'Finance_News'
 'Travel_Ridehailing' 'Social' 'Telecom' 'Entertainment' 'News_finance'
 'Wearable' 'Driver_App']
['Educational' 'Rest' 'Office' 'Ride haling' 'Gaming' 'Finance_Investment'
 'Driver_App']


,app_name,categories_l2,categories_l1
0,ignou e-content,Educational,Educational


### Merge

In [23]:
df_active_customer.head(1)

,channel_host,customer_id,city_name,gender,rapido_age,customer_age,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,fe_count,rr_count,net_count
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0


In [24]:
df_customer_installed_apps_v1.head(1)

,user,app_list
0,656d6c78be45f1519468f640,"[Aha, Amazon Prime Video, Amazon Music, Bookin..."


In [25]:
df_customer_coverage = pd.merge(df_active_customer,
                                df_customer_installed_apps_v1,
                                how='left',
                                left_on=['customer_id'], 
                                right_on=['user']
                                )
df_customer_coverage.head(2)

,channel_host,customer_id,city_name,gender,rapido_age,customer_age,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,fe_count,rr_count,net_count,user,app_list
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,"[Aha, Adobe Scan, Amazon Prime Video, Zomato, ..."
1,android,5d4ed35b55fbf50d45edac48,Hyderabad,MALE,1700.0,24,PHH,80,SUSTENANCE,MEDIUM_INCOME,NO_AFFINITY,UNKNOWN,NPS,5.0,3.0,3.0,5d4ed35b55fbf50d45edac48,"[Amazon Prime Video, Facebook, Flipkart, Googl..."


In [26]:
df_customer_app_list_explode = df_customer_coverage.explode('app_list')
df_customer_app_list_explode['app_list'] = df_customer_app_list_explode['app_list'].str.lower()
df_customer_app_list_explode

,channel_host,customer_id,city_name,gender,rapido_age,customer_age,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,fe_count,rr_count,net_count,user,app_list
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,aha
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,adobe scan
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,amazon prime video
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,zomato
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,wynk music
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2093497,android,65fe7bdfd77e95e12881109e,Hyderabad,MALE,13.0,NA,HH,1,HANDHOLDING,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,UNKNOWN,9.0,1.0,1.0,65fe7bdfd77e95e12881109e,netflix
2093497,android,65fe7bdfd77e95e12881109e,Hyderabad,MALE,13.0,NA,HH,1,HANDHOLDING,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,UNKNOWN,9.0,1.0,1.0,65fe7bdfd77e95e12881109e,ola
2093497,android,65fe7bdfd77e95e12881109e,Hyderabad,MALE,13.0,NA,HH,1,HANDHOLDING,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,UNKNOWN,9.0,1.0,1.0,65fe7bdfd77e95e12881109e,samsung galaxy wearable
2093497,android,65fe7bdfd77e95e12881109e,Hyderabad,MALE,13.0,NA,HH,1,HANDHOLDING,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,UNKNOWN,9.0,1.0,1.0,65fe7bdfd77e95e12881109e,uber


In [30]:
df_customer_app_cat_mapping = pd.merge(df_customer_app_list_explode, 
                                       category_list, 
                                       how='left', 
                                       left_on='app_list', 
                                       right_on='app_name')
df_customer_app_cat_mapping.head(5)

,channel_host,customer_id,city_name,gender,rapido_age,customer_age,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,fe_count,rr_count,net_count,user,app_list,app_name,categories_l2,categories_l1
0,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,aha,aha,Rest,OTT
1,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,adobe scan,adobe scan,Rest,Tools
2,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,amazon prime video,amazon prime video,Rest,OTT
3,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,zomato,zomato,Rest,Delivery_Food
4,android,60766b5ecde8c60e6fc0af3c,Hyderabad,MALE,1087.0,28,PHH,110,DORMANT,HIGH_INCOME,ONLY_LINK,MEDIUM_DISTANCE,NPS,2.0,1.0,0.0,60766b5ecde8c60e6fc0af3c,zomato,zomato,Rest,Delivery_Food


## Analysis

### Customer-wise App list

In [31]:
total_customers = df_customer_coverage.customer_id.nunique()
total_customers

2089119

In [32]:

df_temp = df_customer_app_list_explode\
            .groupby(['app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
print(df_customer_app_list_explode.app_list.nunique())
df_temp

219


,app_list,customers,%
0,google maps,1615635,77.34
1,youtube,1612669,77.19
2,gmail,1612123,77.17
3,whatsapp,1578912,75.58
4,google photos,1543422,73.88
5,instagram,1280854,61.31
6,facebook,1095094,52.42
7,truecaller,1088365,52.10
8,paytm money,1028638,49.24
9,flipkart,977185,46.77


In [33]:
df_customer_mapped = df_customer_app_cat_mapping\
                        .groupby(['customer_id'])\
                        .agg(app_list = ('app_list', lambda x: list(set(x))),
                             app_count = ('app_list', 'nunique'),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
                             categories_l1_count = ('categories_l1', 'nunique'),
                             categories_l2 = ('categories_l2', lambda x: list(set(x))),
                             categories_l2_count = ('categories_l2', 'nunique')
                            )\
                        .reset_index()

print(df_customer_mapped.shape)
df_customer_mapped.head(5)

(2089119, 7)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,5737c6d3ddbec2203f7332ba,"[flipkart, twitter, google maps, facebook, ola...",22,"[Delivery_Food, Office, Tools, OTT, Travel_Rid...",8,"[Rest, Office, Ride haling]",3
1,573f28f39b0ffc283676efee,"[flipkart, sonyliv, slack, google maps, micros...",53,"[Delivery_Food, Office, Health, Tools, Finance...",16,"[Rest, Office, Finance_Investment, Ride haling]",4
2,573f28f39b0ffc283676f013,[nan],0,[nan],0,[nan],0
3,573f28f39b0ffc283676f08f,[nan],0,[nan],0,[nan],0
4,573f28f39b0ffc283676f0a1,"[flipkart, wynk music, google maps, twitter, l...",43,"[Social, Delivery_Food, Office, Educational, T...",13,"[Educational, Office, Finance_Investment, Rest...",5


#### Distribution check on categories

In [34]:
df_temp = df_customer_app_cat_mapping\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list', lambda x: list(set(x))),
                 app_count = ('app_list', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,app_count,customers,%
0,Tools,"[ms edge, microsoft onenote, google maps, gmai...",23,1616283,77.37
1,Social,"[snapchat, pinterest, hike, wechat, roposo, tw...",18,1614195,77.27
2,OTT,"[disney+ hotstar, sonyliv, alt balaji, voot, n...",9,1302594,62.35
3,Ecommerce,"[myntra, meesho, amazon shopping, zivame, flip...",14,1246760,59.68
4,Finance_Transactions,"[paytm money, mobikwik, axis mobile, freecharg...",5,1107020,52.99
5,Travel_Ridehailing,"[blusmart, driveu driver app, namma yatri, qui...",9,1073581,51.39
6,Delivery_Food,"[swiggy, dunzo, uber eats, zomato]",4,1061120,50.79
7,Streaming_Music,"[soundcloud, spotify, wynk music, hungama musi...",6,1020987,48.87
8,Office,"[jira, google analytics, slack, onedrive, trel...",23,869355,41.61
9,News,"[feedly, the indian express, google news]",3,556751,26.65


In [35]:
df_temp = df_customer_app_cat_mapping\
            .groupby(['categories_l2'])\
            .agg(app_list = ('app_list', lambda x: list(set(x))),
                 app_count = ('app_list', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,app_count,customers,%
0,Rest,"[myfitnesspal, flipkart, nfl, fiton, dunzo, fi...",129,1616312,77.37
1,Ride haling,"[blusmart, driveu driver app, namma yatri, qui...",9,1073581,51.39
2,Office,"[jira, google analytics, slack, onedrive, trel...",23,869355,41.61
3,Finance_Investment,"[groww, nps, kite by zerodha, namma partner, h...",6,429819,20.57
4,Gaming,"[free fire, ludo king]",2,282813,13.54
5,Educational,"[happify, fiitjee, brainly, toppr, udemy, cour...",41,280774,13.44
6,Driver_App,"[ola driver app, shadowfax driver app, borzo d...",10,172750,8.27


#### Category-level App usage 

In [36]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['OTT' 'Tools' 'Delivery_Food' 'Streaming_Music' 'Social' 'Ecommerce'
 'Office' 'Travel_Ridehailing' 'Wearable' 'Finance_Transactions' nan
 'News' 'Finance_Investment' 'Educational' 'Travel_Bookings'
 'Entertainment' 'Gaming' 'Driver_App' 'Dating' 'Delivery_Grocery'
 'Finance_News' 'Health' 'Fantasy_Sports' 'Devotional']


In [37]:
print(df_customer_app_cat_mapping['categories_l2'].unique())

['Rest' 'Office' 'Ride haling' nan 'Finance_Investment' 'Educational'
 'Gaming' 'Driver_App']


In [38]:
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,Ride haling,ola,830490,39.75
1,Ride haling,uber,775206,37.11
2,Ride haling,in drive,20353,0.97
3,Ride haling,namma yatri,18170,0.87
4,Ride haling,quick ride,6981,0.33
5,Ride haling,driveu driver app,4392,0.21
6,Ride haling,blusmart,3012,0.14
7,Ride haling,quickride,979,0.05
8,Ride haling,jugnoo,532,0.03
9,Office,zoom,446989,21.40


#### Transform Categories level-2

In [39]:
df_customer_mapped.head(1)

,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,5737c6d3ddbec2203f7332ba,"[flipkart, twitter, google maps, facebook, ola...",22,"[Delivery_Food, Office, Tools, OTT, Travel_Rid...",8,"[Rest, Office, Ride haling]",3


In [40]:
unique_categories = set(category for sublist in df_customer_mapped['categories_l1'] for category in sublist)
# print(unique_categories)

for category in unique_categories:
    df_customer_mapped[category] = df_customer_mapped['categories_l1']\
                                            .apply(lambda x: 1 if category in x else 0)
##When havnig nan last column
df_customer_mapped.drop(df_customer_mapped.columns[-1], axis=1, inplace=True)

In [42]:
df_customer_mapped

,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Entertainment,Social,Travel_Bookings,Office,NaN,Educational,Tools,Gaming,Travel_Ridehailing,Delivery_Grocery,Devotional,Fantasy_Sports,Ecommerce,Finance_Transactions,Streaming_Music,Dating,Finance_Investment,Finance_News,News,Driver_App,Delivery_Food,Health,OTT
0,5737c6d3ddbec2203f7332ba,"[flipkart, twitter, google maps, facebook, ola...",22,"[Delivery_Food, Office, Tools, OTT, Travel_Rid...",8,"[Rest, Office, Ride haling]",3,0,1,0,1,0,0,1,0,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1
1,573f28f39b0ffc283676efee,"[flipkart, sonyliv, slack, google maps, micros...",53,"[Delivery_Food, Office, Health, Tools, Finance...",16,"[Rest, Office, Finance_Investment, Ride haling]",4,1,1,1,1,0,0,1,0,1,1,0,0,1,1,1,0,1,1,0,0,1,1,1
2,573f28f39b0ffc283676f013,[nan],0,[nan],0,[nan],0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,573f28f39b0ffc283676f08f,[nan],0,[nan],0,[nan],0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,573f28f39b0ffc283676f0a1,"[flipkart, wynk music, google maps, twitter, l...",43,"[Social, Delivery_Food, Office, Educational, T...",13,"[Educational, Office, Finance_Investment, Rest...",5,1,1,0,1,0,1,1,0,1,0,0,0,1,1,1,0,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2089114,6609aa3ddb0e9c9b8b404bba,"[youtube, google news, ludo king, flipkart, zo...",17,"[Social, Delivery_Food, Office, Tools, Gaming,...",8,"[Rest, Gaming, Office]",3,0,1,0,1,0,0,1,1,0,0,0,0,1,0,1,0,0,0,1,0,1,0,0
2089115,6609aa705a3971e793e3ae7e,[nan],0,[nan],0,[nan],0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2089116,6609aac5db0e9cf9fd405f1d,"[flipkart, google maps, hdfc mobile banking, g...",19,"[Office, Tools, Finance_Investment, OTT, Ecomm...",7,"[Rest, Office, Finance_Investment]",3,0,1,0,1,0,0,1,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,1
2089117,6609ab51a9381bcb0f3b3b50,"[flipkart, sonyliv, wynk music, google maps, f...",26,"[Delivery_Food, Tools, Gaming, OTT, Entertainm...",9,"[Rest, Gaming, Ride haling]",3,1,1,0,0,0,0,1,1,1,0,0,0,1,0,1,0,0,0,0,0,1,0,1


In [44]:
df_customer_mapped.head(5).T

,0,1,2,3,4
customer_id,5737c6d3ddbec2203f7332ba,573f28f39b0ffc283676efee,573f28f39b0ffc283676f013,573f28f39b0ffc283676f08f,573f28f39b0ffc283676f0a1
app_list,"[flipkart, twitter, google maps, facebook, ola...","[flipkart, sonyliv, slack, google maps, micros...",[nan],[nan],"[flipkart, wynk music, google maps, twitter, l..."
app_count,22,53,0,0,43
categories_l1,"[Delivery_Food, Office, Tools, OTT, Travel_Rid...","[Delivery_Food, Office, Health, Tools, Finance...",[nan],[nan],"[Social, Delivery_Food, Office, Educational, T..."
categories_l1_count,8,16,0,0,13
categories_l2,"[Rest, Office, Ride haling]","[Rest, Office, Finance_Investment, Ride haling]",[nan],[nan],"[Educational, Office, Finance_Investment, Rest..."
categories_l2_count,3,4,0,0,5
Entertainment,0,1,0,0,1
Social,1,1,0,0,1
Travel_Bookings,0,1,0,0,0
